A cute little demo showing the simplest usage of minGPT. Configured to run fine on Macbook Air in like a minute.

In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from mingpt.utils import set_seed
set_seed(3407)

In [ ]:
import pickle

class SortDataset(Dataset):
    """ 
    Dataset for the Sort problem. E.g. for problem length 6:
    Input: 0 0 2 1 0 1 -> Output: 0 0 0 1 1 2
    Which will feed into the transformer concatenated as:
    input:  0 0 2 1 0 1 0 0 0 1 1
    output: I I I I I 0 0 0 1 1 2
    where I is "ignore", as the transformer is reading the input sequence
    """

    def __init__(self, split, length=6, num_digits=3):
        assert split in {'train', 'test'}
        self.split = split
        self.length = length
        self.num_digits = num_digits
    
    def __len__(self):
        return 10000 # ...
    
    def get_vocab_size(self):
        return self.num_digits
    
    def get_block_size(self):
        # the length of the sequence that will feed into transformer, 
        # containing concatenated input and the output, but -1 because
        # the transformer starts making predictions at the last input element
        return self.length * 2 - 1

    def __getitem__(self, idx):
        
        # use rejection sampling to generate an input example from the desired split
        while True:
            # generate some random integers
            inp = torch.randint(self.num_digits, size=(self.length,), dtype=torch.long)
            # half of the time let's try to boost the number of examples that 
            # have a large number of repeats, as this is what the model seems to struggle
            # with later in training, and they are kind of rate
            if torch.rand(1).item() < 0.5:
                if inp.unique().nelement() > self.length // 2:
                    # too many unqiue digits, re-sample
                    continue
            # figure out if this generated example is train or test based on its hash
            h = hash(pickle.dumps(inp.tolist()))
            inp_split = 'test' if h % 4 == 0 else 'train' # designate 25% of examples as test
            if inp_split == self.split:
                break # ok
        
        # solve the task: i.e. sort
        sol = torch.sort(inp)[0]

        # concatenate the problem specification and the solution
        cat = torch.cat((inp, sol), dim=0)

        # the inputs to the transformer will be the offset sequence
        x = cat[:-1].clone()
        y = cat[1:].clone()
        # we only want to predict at output locations, mask out the loss at the input locations
        y[:self.length-1] = -1
        return x, y
    
class Multiply(Dataset):
    """ 
    Dataset for the multiply problem. E.g. for problem size 10:
    Input: 9 * 3 -> Output: 27
    Which will feed into the transformer concatenated as:
    input:  9 3 0 0
    output: I I 2 7 
    or 
    input:  2 3 0 0
    output: I I 0 6 
    where I is "ignore", as the transformer is reading the input sequence
    """

    def __init__(self, split, size=10):
        assert split in {'train', 'test'}
        self.split = split
        self.size = size
    
    def __len__(self):
        return 10000 # ...
    
    def get_vocab_size(self):
        return self.size
    
    def get_num_input(self):
        return len(str(self.size-1))*2
    
    def get_num_output(self):
        return len(str((self.size-1) ** 2))

    def get_block_size(self):
        # the length of the sequence that will feed into transformer, 
        # containing concatenated input and the output, but -1 because
        # the transformer starts making predictions at the last input element
        # block size is: number of digits in input (size) (times two) + number of digits in output (size^2)

        return self.get_num_input() + self.get_num_output()

    def __getitem__(self, idx):
            # generate two random integers in [0, size)
        while True:
            a = torch.randint(self.size, (1,), dtype=torch.long).item()
            b = torch.randint(self.size, (1,), dtype=torch.long).item()
            # decide split based on hash so that train/test are disjoint
            h = hash(pickle.dumps((a, b)))
            inp_split = 'test' if h % 4 == 0 else 'train'  # 25% test
            if inp_split == self.split:
                break

        # compute the product
        prod = a * b  # in [0, (size-1)^2], with size=10 => 0..81

        # encode operands as tokens
        a_tok = a
        b_tok = b

        # encode product as padded 2-digit [tens, ones]
        prod_str = f"{prod:02d}"  # e.g. 6 -> "06", 27 -> "27"
        tens = int(prod_str[0])
        ones = int(prod_str[1])

        # input: a b 0 0
        x = torch.tensor([a_tok, b_tok, 0, 0], dtype=torch.long)

        # output targets: I I tens ones  (I = -1)
        y = torch.tensor([-1, -1, tens, ones], dtype=torch.long)

        return x, y

In [ ]:
# print an example instance of the dataset
train_dataset = Multiply('train')
test_dataset = Multiply('test')
x, y = train_dataset[0]
for a, b in zip(x,y):
    print(int(a),int(b))

In [ ]:
for i in range(20): 
    print(train_dataset[i])

In [ ]:
# create a GPT instance
from mingpt.model import GPT

model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()
model = GPT(model_config)

In [ ]:
# create a Trainer object
from mingpt.trainer import Trainer

train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4 # the model we're using is so small that we can go a bit faster
train_config.max_iters = 4000
train_config.num_workers = 0
trainer = Trainer(train_config, model, train_dataset)

In [ ]:
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

trainer.run()

In [ ]:
# now let's perform some evaluation
model.eval();

In [ ]:
def eval_split(trainer, split, max_batches):
    dataset = {'train':train_dataset, 'test':test_dataset}[split]
    # n = train_dataset.length # naugy direct access shrug
    input_len = train_dataset.get_num_input()
    output_len = train_dataset.get_num_output()
    
    results = []
    mistakes_printed_already = 0
    loader = DataLoader(dataset, batch_size=100, num_workers=0, drop_last=False)
    for b, (x, y) in enumerate(loader):
        x = x.to(trainer.device)
        y = y.to(trainer.device)
        # isolate the input pattern alone
        inp = x[:, :input_len]
        sol = y[:, -output_len:]
        # let the model sample the rest of the sequence
        cat = model.generate(inp, output_len, do_sample=False) # using greedy argmax, not sampling
        sol_candidate = cat[:, input_len:] # isolate the filled in sequence
        # compare the predicted sequence to the true sequence
        correct = (sol == sol_candidate).all(1).cpu() # Software 1.0 vs. Software 2.0 fight RIGHT on this line haha
        for i in range(x.size(0)):
            results.append(int(correct[i]))
            if not correct[i] and mistakes_printed_already < 10: # only print up to 10 mistakes to get a sense
                mistakes_printed_already += 1
                print("GPT claims that %s multiplied is %s but gt is %s" % (inp[i].tolist(), sol_candidate[i].tolist(), sol[i].tolist()))
        if max_batches is not None and b+1 >= max_batches:
            break
    rt = torch.tensor(results, dtype=torch.float)
    print("%s final score: %d/%d = %.2f%% correct" % (split, rt.sum(), len(results), 100*rt.mean()))
    return rt.sum()

# run a lot of examples from both train and test through the model and verify the output correctness
with torch.no_grad():
    train_score = eval_split(trainer, 'train', max_batches=50)
    test_score  = eval_split(trainer, 'test',  max_batches=50)

In [ ]:
# let's run a random given sequence through the model as well
input_len = train_dataset.get_num_input()
output_len = train_dataset.get_num_output()
inp = torch.tensor([[5,6]], dtype=torch.long).to(trainer.device)

assert inp[0].nelement() == input_len
with torch.no_grad():
    cat = model.generate(inp, input_len, do_sample=False)
sol = torch.sort(inp[0])[0]
sol_candidate = cat[:, output_len:]
print('input sequence  :', inp.tolist())
print('predicted sorted:', sol_candidate.tolist())
print('gt sort         :', sol.tolist())
print('matches         :', bool((sol == sol_candidate).all()))